# Python 2

## 0. Настройка окружения

In [2]:
import sys
import re
import math
import string
from collections import Counter
from itertools import product

print("Python:", sys.version.split()[0])
print("Все используемые модули входят в стандартную библиотеку")

Python: 3.12.4
Все используемые модули входят в стандартную библиотеку


---
# 1. Числа → биты → байты

**Любой символ или байт можно представить числом.** Например, латинская `A` в ASCII — число 65. Это *один и тот же байт* в разных системах записи:

| Запись | Значение |
|---|---|
| десятичная | `65` |
| шестнадцатеричная | `0x41` |
| двоичная | `0b01000001` |

**Бит** принимает 0 или 1. **Байт** содержит 8 бит, поэтому хранит значения от 0 до 255. Hex-цифра кодирует 4 бита: `0x41` — два полубайта `0100 0001`.

Не путайте **символ** и **байт**: английские символы в UTF-8 обычно занимают 1 байт, а кириллица чаще 2. При анализе «зашифрованных байтов» работаем с `bytes`, а не вслепую с `ord()` русского текста.

In [2]:
letter = "A"
value = ord(letter)
print("Символ:", letter)
print("Десятичное:", value)
print("Hex:", hex(value))
print("Binary (с 8 позициями):", f"{value:08b}")
print("Обратно:", chr(value))
print("Тот же байт:", b"A", "=>", list(b"A"))
print("Кириллица и UTF-8:", "Я".encode("utf-8"), list("Я".encode("utf-8")))

Символ: A
Десятичное: 65
Hex: 0x41
Binary (с 8 позициями): 01000001
Обратно: A
Тот же байт: b'A' => [65]
Кириллица и UTF-8: b'\xd0\xaf' [208, 175]


### Две операции, которые нужны в реверсе постоянно

- `bytes.fromhex('41 42')` превращает HEX-текст в реальные байты `b'AB'`.
- `.hex()` делает обратное: из байтов в шестнадцатеричную строку.

**Почему это полезно:** задания часто дают массив `0x41, 0x42` или строку `4142`, а правильный ответ нужно прочитать как текст.

In [8]:
data = bytes.fromhex("46 4c 41 47")
print("Байты:", data)
print("HEX:", data.hex())
print("Числа:", list(data))
print("Текст (ASCII):", data.decode("ascii"))

Байты: b'FLAG'
HEX: 464c4147
Числа: [70, 76, 65, 71]
Текст (ASCII): FLAG


## 2. Побитовые операции: `&`, `|`, `^`, `~`

Операции выполняются **над соответствующими битами двух чисел**.

| A | B | `A & B` (И) | `A \| B` (ИЛИ) | `A ^ B` (XOR) |
|---:|---:|---:|---:|---:|
| 0 | 0 | 0 | 0 | 0 |
| 0 | 1 | 0 | 1 | 1 |
| 1 | 0 | 0 | 1 | 1 |
| 1 | 1 | 1 | 1 | 0 |

- **AND `&`**: единица, только если обе единицы. Применяется для **маскирования и проверки битов**.
- **OR `|`**: единица, если хотя бы один бит единица. Применяется для **включения битов**.
- **XOR `^`**: единица, когда биты отличаются. Применяется в преобразованиях и шифрах.
- **NOT `~`**: меняет биты местами, но в Python у целых чисел **нет фиксированной ширины**. Поэтому `~5 == -6`, а для 8-битной инверсии используют `(~x) & 0xFF`.

`and`/`or` в Python — **логические** операторы, а `&`/`|` — **побитовые**. Это не одно и то же.

In [9]:
a = 0b1011
b = 0b0101
for op, result in [("AND", a & b), ("OR ", a | b), ("XOR", a ^ b)]:
    print(f"{a:04b} {op} {b:04b} = {result:04b}")

x = 0b10110110
print("Проверяем бит №2:", f"{x & (1 << 2):08b}", "=>", bool(x & (1 << 2)))
print("Включаем бит №0:", f"{x | (1 << 0):08b}")
print("Выключаем бит №2:", f"{x & ~(1 << 2):08b}")
print("Инвертируем 8 бит:", f"{(~x) & 0xFF:08b}")

1011 AND 0101 = 0001
1011 OR  0101 = 1111
1011 XOR 0101 = 1110
Проверяем бит №2: 00000100 => True
Включаем бит №0: 10110111
Выключаем бит №2: 10110010
Инвертируем 8 бит: 01001001


### Почему XOR можно обратить, а AND — не всегда?

С XOR повторное применение того же ключа возвращает исходное число:


`(x ^ k) ^ k == x`


Причина: каждый бит ключа дважды либо **оставляет** исходный бит (ключ 0), либо **переворачивает** его (ключ 1). Два переворота возвращают оригинал.

А при `x & 0` результат **всегда 0**, независимо от исходного бита. Значит, часть информации потеряна; восстановить её без дополнительных условий нельзя. Аналогично `x | 1` всегда равно 1.

In [12]:
x, key = 0x41, 0x20
encoded = x ^ key
decoded = encoded ^ key
print(f"Исходное: {x:#04x} = {x:08b}")
print(f"Ключ:     {key:#04x} = {key:08b}")
print(f"XOR:      {encoded:#04x} = {encoded:08b}")
print(f"Повтор XOR тем же ключом: {decoded:#04x} => {chr(decoded)}")
assert decoded == x
print("Необратимость AND: 0 & 0 =", 0 & 0, ", 1 & 0 =", 1 & 0)

Исходное: 0x41 = 01000001
Ключ:     0x20 = 00100000
XOR:      0x61 = 01100001
Повтор XOR тем же ключом: 0x41 => A
Необратимость AND: 0 & 0 = 0 , 1 & 0 = 0


### Демонстрационная задача №1. Восстановить строку по XOR и известному ключу

**Дано:** `6c 66 6b 6d`. Каждый байт исходной строки заменили на `байт ^ 0x2A`.

**Рассуждение:** XOR обратим тем же ключом. Следовательно, для расшифрования нужно сделать XOR с `0x2A` **для каждого байта**. Обратите внимание: `bytes` при переборе возвращает числа от 0 до 255, а не односимвольные строки.

In [13]:
cipher = bytes.fromhex("6c 66 6b 6d")
#cipher = bytes.fromhex("6c666b6d")
#cipher = b"\x6c\x66\x6b\x6d"
#cipher = bytes([0x6c, 0x66, 0x6b, 0x6d])
#cipher = bytes([108, 102, 107, 109])
#cipher = b"lfkm"
#cipher = "lfkm".encode("ascii")
#cipher = (0x6c666b6d).to_bytes(4, "big")
key = 0x2A
plain = bytes(b ^ key for b in cipher)
for before, after in zip(cipher, plain):
    print(f"{before:02x} ^ {key:02x} = {after:02x} = {chr(after)!r}")
print("Ответ:", plain.decode("ascii"))
assert plain == b"FLAG"

6c ^ 2a = 46 = 'F'
66 ^ 2a = 4c = 'L'
6b ^ 2a = 41 = 'A'
6d ^ 2a = 47 = 'G'
Ответ: FLAG


## 3. Сдвиги и циклические сдвиги

**Обычный сдвиг** `x << n` перемещает биты влево, заполняя свободные позиции нулями; для неотрицательного числа это похоже на умножение на `2 ** n`. `x >> n` двигает вправо (для неотрицательных чисел похоже на целочисленное деление на `2 ** n`).

**Циклический сдвиг (rotate)** работает в фиксированной ширине, например 8 бит: бит, выпавший слева, **возвращается справа**. В Python нет встроенного оператора `rol`, его надо написать.

Для 8 бит:


`ROL8(x, n) = ((x << n) | (x >> (8 - n))) & 0xFF`


`& 0xFF` нужен, потому что `int` в Python не ограничен 8 битами.

In [14]:
def rol8(x: int, n: int) -> int:
    n %= 8
    x &= 0xFF
    return ((x << n) | (x >> (8 - n))) & 0xFF

def ror8(x: int, n: int) -> int:
    n %= 8
    x &= 0xFF
    return ((x >> n) | (x << (8 - n))) & 0xFF

x = 0b10010110
print(f"Начало: {x:08b}")
print(f"Обычный << 3, потом маска: {(x << 3) & 0xFF:08b}")
y = rol8(x, 3)
print(f"ROL8 на 3: {y:08b}")
print(f"ROR8 обратно: {ror8(y, 3):08b}")
assert ror8(rol8(x, 3), 3) == x

Начало: 10010110
Обычный << 3, потом маска: 10110000
ROL8 на 3: 10110100
ROR8 обратно: 10010110


### Демонстрационная задача №2. Обратить циклический сдвиг

**Алгоритм:** `out = rol8(byte ^ 0x5A, 3)`. Известно `out = 0x91`. Найти `byte`.

**Идём в обратном порядке:** сначала отменяем последний шаг `ROL8` через `ROR8`; потом отменяем XOR повторным XOR с `0x5A`.

In [16]:
out = 0x91
before_rotate = ror8(out, 3)
original = before_rotate ^ 0x5A
print(f"Выход:            {out:08b}")
print(f"После обратного ROR: {before_rotate:08b}")
print(f"После обратного XOR: {original:08b} ({original:#04x})")
assert rol8(original ^ 0x5A, 3) == out

Выход:            10010001
После обратного ROR: 00110010
После обратного XOR: 01101000 (0x68)


---
# 4. Исторические и простые подстановочные шифры

**Подстановка** заменяет одни символы другими по правилу. В шифре Цезаря каждая буква сдвигается на `k` позиций в алфавите. По кругу: после `Z` снова `A`.

Пусть `A=0, B=1, ..., Z=25`. Тогда


`E_k(x) = (x + k) % 26` — шифрование; `D_k(y) = (y - k) % 26` — расшифрование.


Почему `% 26`? В английском алфавите 26 букв, после последней позиции возвращаемся к первой. Это шифр для **учебного анализа**, а не защита секретов: всего 26 сдвигов.

In [17]:
ALPHABET = string.ascii_uppercase

def caesar(text: str, shift: int) -> str:
    result = []
    for char in text.upper():
        if char in ALPHABET:
            index = ord(char) - ord("A")
            result.append(chr(ord("A") + (index + shift) % 26))
        else:
            result.append(char)   # пробелы и знаки не меняем
    return "".join(result)

plain = "CTF IS FUN"
encrypted = caesar(plain, 3)
print("Вход:", plain)
print("Ключ 3:", encrypted)
print("Расшифрование (-3):", caesar(encrypted, -3))
assert caesar(encrypted, -3) == plain

Вход: CTF IS FUN
Ключ 3: FWI LV IXQ
Расшифрование (-3): CTF IS FUN


### Демонстрационная задача №3. Шифр Цезаря, ключ неизвестен

**Дано:** `KHOOR ZRUOG`. Мы не знаем сдвиг, зато число вариантов мало. Можно вывести **все 26 кандидатов** и выбрать осмысленный текст. Так делают с простыми учебными схемами: не всегда требуется сложная математика, иногда достаточно дешёвого перебора.

In [19]:
cipher = "KHOOR ZRUOG"
for key in range(26):
    candidate = caesar(cipher, -key)
    
    print(f"ключ={key:2d}: {candidate}")
print("Вывод: ключ 3 ->", caesar(cipher, -3))

ключ= 0: KHOOR ZRUOG
ключ= 1: JGNNQ YQTNF
ключ= 2: IFMMP XPSME
ключ= 3: HELLO WORLD
ключ= 4: GDKKN VNQKC
ключ= 5: FCJJM UMPJB
ключ= 6: EBIIL TLOIA
ключ= 7: DAHHK SKNHZ
ключ= 8: CZGGJ RJMGY
ключ= 9: BYFFI QILFX
ключ=10: AXEEH PHKEW
ключ=11: ZWDDG OGJDV
ключ=12: YVCCF NFICU
ключ=13: XUBBE MEHBT
ключ=14: WTAAD LDGAS
ключ=15: VSZZC KCFZR
ключ=16: URYYB JBEYQ
ключ=17: TQXXA IADXP
ключ=18: SPWWZ HZCWO
ключ=19: ROVVY GYBVN
ключ=20: QNUUX FXAUM
ключ=21: PMTTW EWZTL
ключ=22: OLSSV DVYSK
ключ=23: NKRRU CUXRJ
ключ=24: MJQQT BTWQI
ключ=25: LIPPS ASVPH
Вывод: ключ 3 -> HELLO WORLD


### Простой шифр замены и важное ограничение

Цезарь — **частный случай** подстановки: одинаковый сдвиг для всех букв. В общей *моноалфавитной замене* каждой букве ставится в соответствие другая буква (взаимно-однозначно), но это уже не один сдвиг.

Если таблица подстановки известна, расшифрование — просто **обратить словарь**. Если неизвестна — простой перебор 26 сдвигов уже не подходит; можно анализировать частоты и закономерности, но это не даёт гарантированного ответа на коротком тексте.

In [20]:
mapping = {"A": "Q", "B": "W", "C": "E", "D": "R", "E": "T"}
reverse_mapping = {new: old for old, new in mapping.items()}
source = "ABCDE"
coded = "".join(mapping[c] for c in source)
decoded = "".join(reverse_mapping[c] for c in coded)
print("Таблица:", mapping)
print("Исходное:", source, "->", coded, "->", decoded)
assert decoded == source

Таблица: {'A': 'Q', 'B': 'W', 'C': 'E', 'D': 'R', 'E': 'T'}
Исходное: ABCDE -> QWERT -> ABCDE


---
# 5. Регулярные выражения (`re`)

Регулярное выражение (**regex**) — это **шаблон для поиска фрагментов текста**. Оно не «понимает смысл» строки: проверяет, подходит ли последовательность символов под заданный шаблон.

| Элемент | Значение | Пример |
|---|---|---|
| `\d` | цифра (в Python по умолчанию Unicode) | `\d+` — несколько цифр |
| `\w` | буква, цифра или `_` (Unicode по умолчанию) | `\w+` |
| `\s` | пробельный символ | пробел, табуляция, перенос |
| `.` | любой символ, кроме `\n` по умолчанию | `a.c` |
| `[A-Z]` | латинская прописная буква | `[A-Z]+` |
| `+` | один или более повторов | `\d+` |
| `*` | ноль или более повторов | `.*` |
| `?` | ноль или один повтор | `colou?r` |
| `^` и `$` | начало и конец строки (с нюансами флагов) | `^id=` |
| `( ... )` | захватывающая группа | `id=(\d+)` |
| `(?: ... )` | незахватывающая группа | `(?:ab)+` |

Пишите шаблоны как `r"..."` — это **raw-строки**, в которых обратный слеш не становится Python-экранированием. Например, `r"\d+"`.

### `search`, `findall`, `sub`: три главных действия

- `re.search(pattern, text)` — **первое совпадение где-нибудь в тексте** или `None`; у результата можно вызвать `.group()`.
- `re.findall(pattern, text)` — **все неперекрывающиеся совпадения**; если в шаблоне есть захватывающие группы, результат содержит **только группы**.
- `re.sub(pattern, replacement, text)` — **заменить** все подходящие фрагменты.

Частая ошибка — считать, что `findall` всегда возвращает полные совпадения. Это правда **только если захватывающих групп нет**.

In [3]:
text = "user=alice; id=42; status=ok; id=407; id=9"
match = re.search(r"id=(\d+)", text)
print("Первое полное совпадение:", match.group(0))
print("Только цифры (группа 1):", match.group(1))
print("Все полные совпадения:", re.findall(r"id=\d+", text))
print("Все значения группы:", re.findall(r"id=(\d+)", text))
print("Замена:", re.sub(r"id=\d+", "id=***", text))

Первое полное совпадение: id=42
Только цифры (группа 1): 42
Все полные совпадения: ['id=42', 'id=407', 'id=9']
Все значения группы: ['42', '407', '9']
Замена: user=alice; id=***; status=ok; id=***; id=***


### Как работают группы в регулярных выражениях — с самого начала

**Группа** — это часть регулярного выражения в круглых скобках `(...)`. Она позволяет не только найти совпадение целиком, но и **достать из него нужный кусочек**.

Допустим, в строке есть `id=42`:

- `r"id=\d+"` найдёт `id=42` целиком;
- `r"id=(\d+)"` тоже найдёт `id=42` целиком, **но запомнит отдельно `42`**;
- `\d` обозначает цифру, `+` — одно или более повторений; круглые скобки выделяют группу.

**Номера групп в `re.search`:** `group(0)` — всё совпадение; `group(1)` — содержимое первой пары `(...)`; `group(2)` — второй и так далее. Нумерация захватывающих групп начинается **с 1**, слева направо по открывающим скобкам.

Например, для выражения `r"user:(\w+) score:(\d+)"` первая группа — имя, вторая — баллы. `\w+` означает один или более «символов слова» (буквы, цифры, подчёркивание). Значения групп возвращаются **строками**, даже если выглядят как числа.

In [ ]:
import re

text = "user=alice; id=42; status=ok"
without_group = re.search(r"id=\d+", text)
with_group = re.search(r"id=(\d+)", text)

print("Без группы:", without_group.group(0))  # id=42
print("Всё совпадение (группа 0):", with_group.group(0))  # id=42
print("Только цифры (группа 1):", with_group.group(1))  # 42

# У pattern без (...) нет группы 1: попытка group(1) вызвала бы IndexError.

### Несколько групп: вытащить разные поля из одного совпадения

Для строки `user:anna score:18` шаблон `r"user:(\w+) score:(\d+)"` выделяет две группы:

- `group(0)` → `user:anna score:18` — всё найденное;
- `group(1)` → `anna` — первая группа `(...)`;
- `group(2)` → `18` — вторая группа `(...)`.

Это удобно при анализе строк журналов, параметров и форматов учебных задач. Если `re.search()` ничего не нашёл, он вернёт `None`: перед `.group()` в общем случае нужно проверить результат.

In [4]:
text = "user:anna score:18"
match = re.search(r"user:(\w+) score:(\d+)", text)
if match is not None:
    print("group(0):", match.group(0))
    print("group(1):", match.group(1))
    print("group(2):", match.group(2))
    print("Баллы как число:", int(match.group(2)))

group(0): user:anna score:18
group(1): anna
group(2): 18
Баллы как число: 18


### Как группы влияют на `re.findall()`

В отличие от `re.search()`, `findall()` возвращает **список результатов**:

| Шаблон | Что вернёт `findall()` |
|---|---|
| `r"id=\d+"` | Полные совпадения: `['id=42', 'id=407', 'id=9']` |
| `r"id=(\d+)"` | Только первую группу каждого совпадения: `['42', '407', '9']` |
| `r"user:(\w+) score:(\d+)"` | Список кортежей из двух групп: `[('anna', '18'), ('ivan', '7')]` |

**Почему так?** Если захватывающих групп нет, возвращаются совпадения целиком. Если одна — её значения. Если две или больше — кортеж значений групп для каждого совпадения.

Если скобки нужны **только для объединения альтернатив**, используем `(?:...)`: это **незахватывающая группа**, она не попадает в возвращаемые значения и не увеличивает номера групп.

In [5]:
text = "id=42; id=407; id=9"
print("Без групп:", re.findall(r"id=\d+", text))
print("С одной группой:", re.findall(r"id=(\d+)", text))

scores = "user:anna score:18 | user:ivan score:7"
print("С двумя группами:", re.findall(r"user:(\w+) score:(\d+)", scores))

animals = "cats dogs cat"
print("Незахватывающая группа:", re.findall(r"(?:cat|dog)s?", animals))
# (?:cat|dog) означает «cat или dog», s? — необязательная буква s.

Без групп: ['id=42', 'id=407', 'id=9']
С одной группой: ['42', '407', '9']
С двумя группами: [('anna', '18'), ('ivan', '7')]
Незахватывающая группа: ['cats', 'dogs', 'cat']


### Демонстрационная задача №4. Извлечь имена и баллы

**Строка:** `user:anna score:18 | user:ivan score:7`.

Нужны пары `(имя, баллы)`:

1. Найти литерал `user:`.
2. Захватить имя `(` + `\w+` + `)`.
3. Пропустить пробелы `\s+`.
4. Найти литерал `score:` и захватить цифры `(` + `\d+` + `)`.
5. `findall` вернёт **кортежи строк**, так что для арифметики преобразуем числа через `int()`.

In [6]:
text = "user:anna score:18 | user:ivan score:7"
pattern = r"user:(\w+)\s+score:(\d+)"
raw_pairs = re.findall(pattern, text)
pairs = [(name, int(score)) for name, score in raw_pairs]
print("Найдено:", raw_pairs)
print("После преобразования:", pairs)
print("Всего баллов:", sum(score for _, score in pairs))
assert pairs == [("anna", 18), ("ivan", 7)]

Найдено: [('anna', '18'), ('ivan', '7')]
После преобразования: [('anna', 18), ('ivan', 7)]
Всего баллов: 25


### Жадность `.*` и ленивость `.*?`

По умолчанию квантификаторы `*` и `+` **жадные**: стремятся захватить максимально длинную подходящую часть, **но** оставляют место для остального шаблона. Модификатор `?` после квантификатора делает его **нежадным**.

Строка: `<b>one</b><b>two</b>`

- `<b>.*</b>` начинает с первого `<b>` и заканчивает **последним** `</b>`.
- `<b>.*?</b>` останавливается на **первом** подходящем `</b>`, затем ищет следующее совпадение.

**Важно:** регулярные выражения удобны для учебных строк и ограниченных форматов; для настоящего HTML обычно используют парсер, а не `.*?`.

In [7]:
html = "<b>one</b><b>two</b>"
print("Жадный:", re.findall(r"<b>.*</b>", html))
print("Нежадный:", re.findall(r"<b>.*?</b>", html))
print("Только содержимое:", re.findall(r"<b>(.*?)</b>", html))

Жадный: ['<b>one</b><b>two</b>']
Нежадный: ['<b>one</b>', '<b>two</b>']
Только содержимое: ['one', 'two']


### `re.sub` с группами: изменить только нужный фрагмент

Скобки помогают **сохранить часть совпадения** при замене. Ссылки вида ``\1`` и ``\2`` относятся к группам 1 и 2. Надёжный вариант — использовать функцию `lambda match: ...`, чтобы обработать группу как число.

**Демонстрация:** увеличить каждый счёт на 5, не меняя имя игрока.

In [8]:
source = "user:anna score:18 | user:ivan score:7"
updated = re.sub(
    r"(score:)(\d+)",
    lambda m: m.group(1) + str(int(m.group(2)) + 5),
    source,
)
print("Было:", source)
print("Стало:", updated)

Было: user:anna score:18 | user:ivan score:7
Стало: user:anna score:23 | user:ivan score:12


---
# 6. Математика для реверса: степени, модуль, простые числа

## 6.1. Остаток от деления `%` и степень `**`

- `a ** b` — возвести `a` в степень `b`. Не путать с `a ^ b` (XOR).
- `a % m` — остаток от деления на `m`, если `m > 0`, то от 0 до `m - 1`.
- `pow(a, b, m)` — вычислить `(a ** b) % m` **без построения огромного числа `a**b`** (при `b >= 0`, `m > 0`).

Если представить остатки как часы, то `(10+5) % 12 = 3`: после 12 снова идёт 0. Поэтому модульная арифметика часто встречается в шифрах, генераторах и проверках.

In [9]:
print("2 ** 10 =", 2 ** 10)
print("1024 % 7 =", 1024 % 7)
print("pow(2, 10, 7) =", pow(2, 10, 7))
print("^ — это XOR, а не степень: 2 ^ 10 =", 2 ^ 10)
print("(-3) % 7 =", (-3) % 7)
assert pow(2, 10, 7) == 2

2 ** 10 = 1024
1024 % 7 = 2
pow(2, 10, 7) = 2
^ — это XOR, а не степень: 2 ^ 10 = 8
(-3) % 7 = 4


## 6.2. Как работает быстрое возведение в степень по модулю

Нужно найти `3 ** 13 % 7`. Разложим 13 на степени двойки: `13 = 8 + 4 + 1`. Вычислим остатки квадратов:

- `3 ** 1 % 7 == 3`
- `3 ** 2 % 7 == 2`
- `3 ** 4 % 7 == 4`
- `3 ** 8 % 7 == 2`

Теперь перемножим нужные остатки: `(2 * 4 * 3) % 7 == 3`.

Так устроена идея **двоичного возведения в степень** — количество шагов растёт примерно как число бит в показателе, а не как сам показатель.

In [10]:
def modular_power_verbose(base: int, exponent: int, modulus: int) -> int:
    if exponent < 0 or modulus <= 0:
        raise ValueError("В этом примере показатель >= 0, модуль > 0")
    result = 1
    base %= modulus
    step = 0
    while exponent:
        print(f"Шаг {step}: exp={exponent:>2} ({exponent:b}₂), base={base}, result={result}")
        if exponent & 1:
            result = result * base % modulus
            print("   Младший бит 1 -> умножаем result на base по модулю")
        base = base * base % modulus
        exponent >>= 1
        step += 1
    return result % modulus

answer = modular_power_verbose(3, 13, 7)
print("Ответ:", answer, "| встроенная проверка:", pow(3, 13, 7))
assert answer == pow(3, 13, 7) == 3

Шаг 0: exp=13 (1101₂), base=3, result=1
   Младший бит 1 -> умножаем result на base по модулю
Шаг 1: exp= 6 (110₂), base=2, result=3
Шаг 2: exp= 3 (11₂), base=4, result=3
   Младший бит 1 -> умножаем result на base по модулю
Шаг 3: exp= 1 (1₂), base=2, result=5
   Младший бит 1 -> умножаем result на base по модулю
Ответ: 3 | встроенная проверка: 3


## 6.3. НОД, взаимная простота, обратный элемент

**НОД** двух чисел — наибольшее число, на которое делятся оба. Например, НОД(12, 18) = 6. Если `gcd(a, m) == 1`, числа **взаимно простые**.

**Обратный по модулю** к `a` — число `a_inverse`, которое удовлетворяет:


`(a * inverse_a) % m == 1`


Например, `7 * 15 % 26 == 1`, поэтому 15 — обратный элемент к 7 по модулю 26.

**Почему это нужно:** если `y = (a*x+b) % m`, то вычесть `b` легко, а убрать умножение на `a` можно через обратный элемент **только когда** `gcd(a,m)=1`. Делить на `a` обычным `/` по модулю нельзя.

In [11]:
print("gcd(12, 18) =", math.gcd(12, 18))
print("gcd(7, 26) =", math.gcd(7, 26))
print("Обратный к 7 по модулю 26:", pow(7, -1, 26))
print("Проверка:", (7 * pow(7, -1, 26)) % 26)
try:
    pow(6, -1, 26)
except ValueError as exc:
    print("У 6 нет обратного по модулю 26:", type(exc).__name__)

gcd(12, 18) = 6
gcd(7, 26) = 1
Обратный к 7 по модулю 26: 15
Проверка: 1
У 6 нет обратного по модулю 26: ValueError


### Демонстрационная задача №5. Решить сравнение, не перебирая все числа

**Дано:** `(7*x + 3) % 26 == 15`. Найти `x` среди `0..25`.

1. Вычесть 3 из обеих сторон по модулю 26: `7*x ≡ 12 (mod 26)`.
2. Найти число, обратное к 7: это 15, ведь `7*15 % 26 == 1`.
3. Умножить сравнение на 15: `x ≡ 12*15 ≡ 24 (mod 26)`.
4. Проверить найденный ответ в исходной проверке.

Важная оговорка: **среди всех целых** решений бесконечно много: `x = 24 + 26*k`; среди 0..25 — одно.

In [12]:
a, b, y, modulus = 7, 3, 15, 26
inverse_a = pow(a, -1, modulus)
x = ((y - b) * inverse_a) % modulus
print("Обратный элемент:", inverse_a)
print("Восстановили x:", x)
print("Проверка:", (a * x + b) % modulus)
print("Перебор для контроля:", [i for i in range(modulus) if (a*i+b) % modulus == y])
assert x == 24

Обратный элемент: 15
Восстановили x: 24
Проверка: 15
Перебор для контроля: [24]


## 6.4. Простые числа и комбинаторика

**Простое число** больше 1 делится без остатка только на 1 и на себя. Чтобы проверить, простое ли `n`, достаточно поискать делители до `sqrt(n)`. Если больший делитель существовал бы, то у него был бы парный меньший делитель.

**Комбинаторика** позволяет оценить стоимость перебора:

- у однобайтового ключа — `2 ** 8 == 256` вариантов;
- у двухбайтового — `2 ** 16 == 65536`;
- у строки из 4 букв алфавита `ABC` — `3 ** 4 == 81` комбинация.

**Ключевой вопрос при реверсе:** много ли возможных входов? Если их 26 или 256 — перебор разумен. Если вариантов миллиарды, ищем математическое сокращение или дополнительную информацию. Количество комбинаций **не равно** реальной сложности автоматически: важно, сколько стоит одна проверка.

In [ ]:
def is_prime(n: int) -> bool:
    if n < 2:
        return False
    for d in range(2, math.isqrt(n) + 1):
        if n % d == 0:
            return False
    return True

print("Простые до 30:", [n for n in range(30) if is_prime(n)])
print("Возможных 1-байтовых ключей:", 2**8)
print("Возможных 2-байтовых ключей:", 2**16)
print("Комбинации из ABC длины 4:", 3**4)
print("Первые 6:", ["".join(chars) for chars in list(product("ABC", repeat=4))[:6]])

---
# 7. Анализ и реверс алгоритмов: главный блок

## 7.1. Обратить цепочку операций

**Условие:** неизвестный байт `x` превратили в `y = ((x + 5) ^ 0x12)`. Известно, что `y=84`.

**Прямо:** прибавили 5 → применили XOR с 18.  
**Обратно:** применили XOR с 18 → вычли 5.

Почему именно в таком порядке? Если надеть носки, а потом ботинки, то снимать надо сначала ботинки, потом носки. Преобразования ведут себя так же.

In [ ]:
def forward_simple(x):
    return (x + 5) ^ 0x12

y = 84
x = (y ^ 0x12) - 5
print("Дано y:", y)
print("Отмена XOR:", y ^ 0x12)
print("Отмена +5:", x)
print("Проверяем прямой функцией:", forward_simple(x))
assert x == 65 and forward_simple(x) == y

## 7.2. Сложный байтовый алгоритм: порядок действий и переполнение

В задачах на бинарные операции часто есть `& 0xFF`: арифметика выполняется **по модулю 256**, словно регистр имеет ширину один байт.

Например:

```python
b = ((x ^ 0x35) + 17) & 0xFF
out = rol8(b, 2)
```

Восстанавливаем в обратном порядке:

1. `ror8(out, 2)` отменяет вращение;
2. `(value - 17) & 0xFF` отменяет прибавление с переполнением;
3. `value ^ 0x35` отменяет XOR.

**Подводный камень:** без `& 0xFF` сложение в Python не переполняется, поэтому прямой и обратный алгоритмы могут не совпасть с 8-битной логикой задания.

In [ ]:
def encode_byte(x: int) -> int:
    b = ((x ^ 0x35) + 17) & 0xFF
    return rol8(b, 2)

def decode_byte(out: int) -> int:
    b = ror8(out, 2)
    b = (b - 17) & 0xFF
    return b ^ 0x35

source = b"CTF"
cipher = bytes(encode_byte(b) for b in source)
restored = bytes(decode_byte(b) for b in cipher)
print("Исходное:", source)
print("Шифртекст hex:", cipher.hex())
print("Обратный алгоритм:", restored)
assert restored == source
assert all(decode_byte(encode_byte(i)) == i for i in range(256))

## 7.3. Необратимый шаг: почему нельзя просто «отменить AND»

**Дано:** `y = x & 0x0F`, причём `y = 0x06`. Какие `x` подходят?

Маска оставляет только 4 младших бита. Значит, старшие четыре бита неизвестны: есть **16 разных исходных байтов**, имеющих младшую часть `0110`.

Если известно, что исходный байт — заглавная латинская буква, можно отфильтровать кандидатов по `0x41..0x5A`. Это пример **сочетания анализа алгоритма с ограничениями на вход**.

In [ ]:
y = 0x06
candidates = [x for x in range(256) if (x & 0x0F) == y]
uppercase = [x for x in candidates if 65 <= x <= 90]
print("Всего кандидатов:", len(candidates))
print("Их hex:", [f"{x:02x}" for x in candidates])
print("Заглавные ASCII:", [chr(x) for x in uppercase])
assert len(candidates) == 16

## 7.4. Секретный, но слабый параметр: XOR с одним байтом

Предположим, каждый байт текста XOR-или с **одним неизвестным ключом** в диапазоне `0..255`. Это всего 256 вариантов. Часто можно определить ключ по **известному фрагменту текста** (known plaintext).

Если известны исходный байт `p` и зашифрованный `c`, то ключ `k = p ^ c`.

В отличие от стойкой современной криптографии, **повторяющийся однобайтовый XOR** легко раскрывается при известном открытом тексте. Не путать его с одноразовым блокнотом, где ключ случаен, секретен и используется однократно.

In [ ]:
secret_key = 0x2A
original = b"FLAG{DEMO}"
encrypted = bytes(b ^ secret_key for b in original)

# Из формата задачи известно, что сообщение начинается с b"FLAG{"
known_prefix = b"FLAG{"
recovered_key = encrypted[0] ^ known_prefix[0]
print("Зашифрованные байты:", encrypted.hex())
print("Вычисленный ключ:", hex(recovered_key))
print("Проверяем ключ по ВСЕМ байтам известного префикса:",
      all((encrypted[i] ^ recovered_key) == known_prefix[i] for i in range(len(known_prefix))))
print("Расшифрованное сообщение:", bytes(c ^ recovered_key for c in encrypted))
assert recovered_key == secret_key

## 7.5. Разбор незнакомого кода: поиск входа по условию

Представим, что в олимпиадном задании есть **проверяющая функция**. Она скрывает ожидаемое значение с помощью нескольких преобразований.

```python
def check(x):
    return (((x ^ 0x21) + 9) % 256) == 0x53
```

Нам нужно не подобрать пароль «наугад», а найти такое `x`, чтобы функция вернула `True`.

**Пошагово:**

- после сложения и остатка получилось `0x53`;
- отменяем `+ 9` по модулю 256;
- отменяем XOR с `0x21`;
- проверяем вход через исходную `check`.

In [ ]:
def check(x: int) -> bool:
    return (((x ^ 0x21) + 9) % 256) == 0x53

target = 0x53
before_addition = (target - 9) % 256
solution = before_addition ^ 0x21
print("После отмены сложения:", hex(before_addition))
print("После отмены XOR:", hex(solution), "=", solution, "ASCII:", chr(solution))
print("Проверка:", check(solution))
print("Контроль полным перебором:", [x for x in range(256) if check(x)])
assert check(solution)

## 7.6. Математический реверс + комбинаторика: когда есть несколько решений

Допустим, функция возвращает `y = (6*x+4) % 26`, а нам дано `y=10`.

В отличие от предыдущего примера, `gcd(6,26)=2`, поэтому обратного к 6 по модулю 26 **нет**. В этом случае нельзя бездумно делать `pow(6, -1, 26)`.

Для маленького пространства `0..25` разумно **перебрать варианты**. Обратите внимание: решений может быть несколько. Это не ошибка перебора — функция действительно не сохраняет уникальность входа.

In [ ]:
def weak_check(x: int) -> bool:
    return (6*x + 4) % 26 == 10

print("НОД(6,26):", math.gcd(6, 26))
solutions = [x for x in range(26) if weak_check(x)]
print("Все решения среди 0..25:", solutions)
print("Проверки:", [(x, (6*x+4)%26) for x in solutions])
assert solutions == [1, 14]

## 7.7. Смешанный пример: текст → байты → преобразование → восстановление

Финальная демонстрация объединяет несколько приёмов. Дана короткая учебная процедура:

1. Преобразует строку ASCII в байты.
2. Для байта с индексом `i` вычисляет `(b + i) & 0xFF`.
3. Применяет XOR с ключом `0x13`.
4. Выдаёт HEX-строку.

Нужно **написать обратное преобразование**, а затем доказать правильность повторным шифрованием. Важно: индекс не является ключом шифрования, но влияет на каждый байт — забыть его нельзя.

In [ ]:
def pack_message(text: str) -> str:
    source = text.encode("ascii")
    return bytes((((b + i) & 0xFF) ^ 0x13) for i, b in enumerate(source)).hex()

def unpack_message(hex_text: str) -> str:
    cipher = bytes.fromhex(hex_text)
    source = bytes((((c ^ 0x13) - i) & 0xFF) for i, c in enumerate(cipher))
    return source.decode("ascii")

input_text = "REVERSING"
hex_value = pack_message(input_text)
print("Шаги для первого символа:")
first = ord(input_text[0])
print(f"Символ {input_text[0]!r} -> {first} -> +0 -> {first} -> XOR 0x13 -> {first ^ 0x13:02x}")
print("Весь шифртекст:", hex_value)
print("Восстановленный текст:", unpack_message(hex_value))
print("Повторная упаковка совпадает:", pack_message(unpack_message(hex_value)) == hex_value)
assert unpack_message(hex_value) == input_text

## 7.8. Дополнительно: как быстро понять, что за преобразование перед нами

При чтении неизвестного Python-кода удобно узнавать **характерные признаки**:

| В коде встретилось | Какой вопрос задать |
|---|---|
| `^ key` | Известен ли ключ? Можно ли применить XOR ещё раз? |
| `& 0xFF` | Это 8-битное преобразование? Где происходит переполнение? |
| `<<`, `>>` | Обычный сдвиг или циклический? Сколько бит? |
| `% m` | В каком диапазоне результат? Есть ли обратный по модулю? |
| `ord`, `chr`, `encode`, `decode` | Где здесь символы, а где числа и байты? |
| `bytes.fromhex` | Вход — HEX-представление или настоящие байты? |
| `re.findall` | Что именно захватывают группы? |
| `range(256)` | Не пытаются ли перебрать все возможные байтовые значения? |

**Принцип:** перед тем как «ломать шифр», нужно понять, что именно было сделано с данными. Очень часто задача решается несколькими обратными действиями.